<a href="https://colab.research.google.com/github/sahaurja/Auto-Pauser-Model/blob/main/Auto_Pause_Transcripts.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Getting Transcripts

In [2]:
pip install youtube-transcript-api

In [5]:
from youtube_transcript_api import YouTubeTranscriptApi

In [23]:
ytt_api = YouTubeTranscriptApi()
video_id = "P-_Nzi_mCRo"  # The video ID from the URL
fetched_transcript = ytt_api.fetch(video_id)

In [42]:
# is iterable
for snippet in fetched_transcript:
    print(snippet)

FetchedTranscriptSnippet(text='when you got started programming in java', start=0.32, duration=2.8)
FetchedTranscriptSnippet(text='i bet somebody just', start=2.0, duration=2.96)
FetchedTranscriptSnippet(text='told you that you had to have this', start=3.12, duration=3.04)
FetchedTranscriptSnippet(text='public static', start=4.96, duration=4.0)
FetchedTranscriptSnippet(text='void main string args and in there is', start=6.16, duration=3.84)
FetchedTranscriptSnippet(text='where you write your code', start=8.96, duration=3.2)
FetchedTranscriptSnippet(text='then i bet you probably thought oh wait', start=10.0, duration=3.28)
FetchedTranscriptSnippet(text='what does that mean wait', start=12.16, duration=3.599)
FetchedTranscriptSnippet(text="wait what's with public static void main", start=13.28, duration=3.12)
FetchedTranscriptSnippet(text='string', start=15.759, duration=1.921)
FetchedTranscriptSnippet(text='what does all that mean and they', start=16.4, duration=3.36)
FetchedTranscriptS

In [21]:
#dictionary version...text, start, duration
# fetched_transcript.to_raw_data()

# for ins in fetched_transcript.to_raw_data():
#     print(ins)

print(fetched_transcript.to_raw_data()[0])

{'text': 'when you got started programming in java', 'start': 0.32, 'duration': 2.8}


Try finding simialrity between consecutive snippets using SpaCy

In [34]:
!python -m spacy download en_core_web_md

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 41.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [35]:
import spacy

In [36]:
nlp = spacy.load("en_core_web_md") #english model

In [37]:
#try with first two snippets
doc1 = nlp(fetched_transcript[0].text)
doc2 = nlp(fetched_transcript[1].text)
print(doc1, "<->", doc2, doc1.similarity(doc2))

when you got started programming in java <-> i bet somebody just 0.7961385846138


In [48]:
#iterate along and check for all vectors
all_pauses = [] #too different
all_cont = [] #similar enough
#threshhopld kind of randomly chosen for now
thresh = 0.5
for i in range(len(fetched_transcript)-1):
  doc1 = nlp(fetched_transcript[i].text)
  doc2 = nlp(fetched_transcript[i+1].text)
  sim = doc1.similarity(doc2)
  if(sim < thresh):
    all_pauses.append({"snippet1": doc1, "snippet2": doc2, "time_end": fetched_transcript[i+1].start, "similarity": sim})
  else:
    all_cont.append({"snippet1": doc1, "snippet2": doc2, "time_end": fetched_transcript[i+1].start, "similarity": sim})

In [45]:
import pandas as pd

In [49]:
pause_data = pd.DataFrame(all_pauses)
cont_data = pd.DataFrame(all_cont)

In [52]:
from tabulate import tabulate


print(tabulate(pause_data, headers='keys', tablefmt='psql'))


+----+------------------------------------------+------------------------------------------+------------+--------------+
|    | snippet1                                 | snippet2                                 |   time_end |   similarity |
|----+------------------------------------------+------------------------------------------+------------+--------------|
|  0 | told you that you had to have this       | public static                            |      4.96  |     0.392455 |
|  1 | wait what's with public static void main | string                                   |     15.759 |     0.213883 |
|  2 | string                                   | what does all that mean and they         |     16.4   |     0.149877 |
|  3 | think about what is happening here is    | that the jre the java runtime            |     47.12  |     0.480614 |
|  4 | that the jre the java runtime            | environment on your computer             |     49.68  |     0.395037 |
|  5 | method                   

In [53]:
from tabulate import tabulate


print(tabulate(cont_data, headers='keys', tablefmt='psql'))


+-----+------------------------------------------+------------------------------------------+------------+--------------+
|     | snippet1                                 | snippet2                                 |   time_end |   similarity |
|-----+------------------------------------------+------------------------------------------+------------+--------------|
|   0 | when you got started programming in java | i bet somebody just                      |      2     |     0.796139 |
|   1 | i bet somebody just                      | told you that you had to have this       |      3.12  |     0.890202 |
|   2 | public static                            | void main string args and in there is    |      6.16  |     0.517583 |
|   3 | void main string args and in there is    | where you write your code                |      8.96  |     0.621429 |
|   4 | where you write your code                | then i bet you probably thought oh wait  |     10     |     0.794467 |
|   5 | then i bet you p